# ДЗ 2. Предобработка и векторизация текстовых данных

Работа с датасетом отзывов «Перекрёсток».

1. **Предобработать тексты**: нижний регистр, удаление пунктуации и лишних символов, токенизация, удаление стоп-слов, лемматизация.
2. **Векторизовать тексты**: Bag of Words (BoW) и TF-IDF.
3. **Сравнить представления**: размерность матриц, разреженность, примеры векторов, отличия BoW и TF-IDF.

Помимо базового конвейера ниже подробно разобран **выбор стоп-слов** под задачу тональности: какие служебные слова оставить (отрицания, оценки), какие частотные слова и шаблонные ярлыки формы отзыва убрать. Каждое решение проверено на данных.

## 0. Установка и импорт зависимостей

In [3]:
%pip install datasets pandas numpy matplotlib seaborn pymorphy3 nltk emoji scikit-learn tqdm -q


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import pandas as pd
import re
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
RANDOM_STATE = 42

## 1. Загрузка данных

Датасет `lapki/perekrestok-reviews` (~642 тыс. русскоязычных отзывов о товарах). Загружаем напрямую с Hugging Face.

In [5]:
df = pd.read_json("hf://datasets/lapki/perekrestok-reviews/perekrestok.jsonl", lines=True)
df.head()

,product_id,product_name,product_category,product_price,review_id,review_author,review_text,rating
0,1261049,Десерт Fit Kit глазированный протеиновый Двойн...,Здоровое питание,213.0,3396923,Милана,Самое вкусное протеиновое печенье с белковым с...,5.0
1,984168,Свеча для торта Paterra Детская Цифра 1,"Дом, дача, книги",97.9,1977331,любовь,"Веселая свечка, но нам пришла розовая, как на ...",5.0
2,984168,Свеча для торта Paterra Детская Цифра 1,"Дом, дача, книги",97.9,1451326,инна,Хорошая яркая свечка,5.0
3,984168,Свеча для торта Paterra Детская Цифра 1,"Дом, дача, книги",97.9,449473,Marina,"Ребенок в восторге от свечек! Она красная, кст...",5.0
4,1271460,Сибас филе охлажденное,"Рыба, икра",1779.0,3444792,Роман,"Хорошая рыба, вкусная, свежая. Жаль, что возят...",5.0


## 2. Очистка данных

Перед предобработкой текста убираем проблемы качества, выявленные в ДЗ 1:

- **пустые отзывы** — бесполезны для текстовой задачи;
- **дубликаты** — удаляем по содержательно-метаданному ключу (**без `review_id`**): оказалось, что один и тот же отзыв выгружен повторно под разными `review_id` (тексты развёрнутые, медиана 35 слов, совпадают дословно вместе с автором/товаром/ценой), поэтому `review_id` ненадёжен как ключ уникальности;
- **некорректный рейтинг** (вне 0–5) и **цена ≤ 0** — обрабатываем на всякий случай (в данных таких нет).

In [6]:
before = len(df)
log = {}

# пустые отзывы
mask_empty = df["review_text"].isna() | (df["review_text"].astype(str).str.strip() == "")
df = df[~mask_empty].copy()
log["удалено пустых отзывов"] = int(mask_empty.sum())

# дубликаты по содержанию+метаданным (без review_id — он у повторных выгрузок разный)
key_cols = ["product_id", "product_name", "product_category", "product_price",
            "review_author", "review_text", "rating"]
dup_mask = df.duplicated(subset=key_cols)
df = df[~dup_mask].copy()
log["удалено дубликатов (ключ без review_id)"] = int(dup_mask.sum())

# некорректный рейтинг
bad_rating = (df["rating"] < 0) | (df["rating"] > 5) | df["rating"].isna()
df = df[~bad_rating].copy()
log["удалено с некорректным рейтингом"] = int(bad_rating.sum())

# цена <= 0 -> NaN, затем заполнить медианой по категории
df.loc[df["product_price"] <= 0, "product_price"] = np.nan
n_price_na = df["product_price"].isna().sum()
df["product_price"] = df["product_price"].fillna(
    df.groupby("product_category")["product_price"].transform("median"))
df["product_price"] = df["product_price"].fillna(df["product_price"].median())
log["цен заполнено медианой"] = int(n_price_na)

df = df.reset_index(drop=True)
log["итог: строк было"] = before
log["итог: строк стало"] = len(df)
log["итог: удалено всего"] = before - len(df)
pd.Series(log)

удалено пустых отзывов                          1
удалено дубликатов (ключ без review_id)     15060
удалено с некорректным рейтингом                0
цен заполнено медианой                          0
итог: строк было                           642682
итог: строк стало                          627621
итог: удалено всего                         15061
dtype: int64

### Приведение рейтинга к целому

Формально `rating` дробный, но фактически шкала целочисленная: на оценки 0–5 приходится ~99.6 % записей, дробные (4.7, 4.3 …) единичны. Округляем к ближайшему целому (порог 0.5), чтобы работать с чистыми классами.

In [7]:
df["rating"] = np.floor(df["rating"].astype(float) + 0.5).clip(0, 5).astype(int)
print(df["rating"].value_counts().sort_index())

rating
0        23
1     16651
2     11153
3     19434
4     93928
5    486432
Name: count, dtype: int64


## 3. Анализ эмодзи

Эмодзи несут тональность, поэтому прежде чем их удалять, оценим: насколько они часты, сколько отзывов состоят **только** из эмодзи и как эмодзи распределены по оценкам.

In [8]:
import emoji as emo

def strip_emoji(t):
    return emo.replace_emoji(str(t), replace=" ")

has_emoji = df["review_text"].apply(lambda t: emo.emoji_count(str(t)) > 0)
print("Доля отзывов с эмодзи:", f"{has_emoji.mean():.2%}")

# разделяем: эмодзи + текст  vs  только эмодзи (после удаления эмодзи не осталось букв)
sub = df.loc[has_emoji, "review_text"]
has_words = sub.apply(lambda t: bool(re.search(r"[a-zа-яё]", strip_emoji(t).lower())))
print("  со словами (эмодзи + текст):", has_words.sum(), f"({has_words.mean():.1%})")
print("  только эмодзи (без слов):   ", (~has_words).sum(), f"({(~has_words).mean():.1%})")
print("\nПримеры «только эмодзи»:", [" ".join(str(t).split()) for t in sub[~has_words].head(5)])

Доля отзывов с эмодзи: 3.63%
  со словами (эмодзи + текст): 22680 (99.7%)
  только эмодзи (без слов):    77 (0.3%)

Примеры «только эмодзи»: ['👍👍👍', '👍', '👌👌👌', '♡♡♡♡♥♥♥♥♡♡♡♡', '👍👍👍']


In [9]:
# распределение оценок у отзывов с эмодзи vs общий фон
cmp = pd.DataFrame({
    "все":      df["rating"].value_counts(normalize=True),
    "с эмодзи": df.loc[has_emoji, "rating"].value_counts(normalize=True),
}).fillna(0).sort_index().round(3)
print(cmp)
print("\nСредний рейтинг: все =", round(df["rating"].mean(), 2),
      "| с эмодзи =", round(df.loc[has_emoji, "rating"].mean(), 2))

          все  с эмодзи
rating                 
0       0.000     0.000
1       0.027     0.014
2       0.018     0.007
3       0.031     0.012
4       0.150     0.141
5       0.775     0.827

Средний рейтинг: все = 4.63 | с эмодзи = 4.76


**Проверка: совпадают ли эмодзи с тональностью текста.** Выведем по 10 коротких (≤5 слов) эмодзи-отзывов на каждый рейтинг — если у 5 эмодзи позитивные, а у 1–2 негативные, значит эмодзи дублируют словесную оценку.

In [10]:
has_emoji = df["review_text"].apply(lambda t: emo.emoji_count(str(t)) > 0)
words_wo_emoji = df["review_text"].apply(
    lambda t: len(emo.replace_emoji(str(t), replace=" ").split()))
mask = has_emoji & (words_wo_emoji <= 5)

pd.set_option("display.max_colwidth", 100)
for r in sorted(df["rating"].unique()):
    sub = df.loc[mask & (df["rating"] == r), "review_text"]
    print("=" * 70)
    print(f"Рейтинг {r} — коротких эмодзи-отзывов: {len(sub)}")
    for t in sub.head(10):
        print("  •", " ".join(str(t).split())[:90])

Рейтинг 0 — коротких эмодзи-отзывов: 0
Рейтинг 1 — коротких эмодзи-отзывов: 62
  • Химический вкус👎
  • Ужасная дрянь👎
  • Это просто ужас. Не берите🤢
  • Очень пересоленые,невозможно есть!🙄
  • Запах просто Жуть 😷😷😷
  • Не приятно пахнет 🤮
  • Отвратительная баклажанная икра!👎
  • Кисляк жуткий👎
  • Кислая, очень 👎
  • Кислая очень👎
Рейтинг 2 — коротких эмодзи-отзывов: 31
  • Кислые🤪
  • Вкусно👍
  • Резиновые, сухие, безвкусные😔
  • Честно говоря, не очень 🥲
  • Ужасный вкус 🤢🤮
  • Колбаса - сплошная химия. 👎
  • Мне вкус не понравился👎
  • Очень тонкие,мне не понравились🙃
  • 😕 Так себе
  • Странная говядина со вкусом курицы🤔
Рейтинг 3 — коротких эмодзи-отзывов: 38
  • Не хрустит! На троечку перец 😏
  • Тошнотворный вкус, утром чистить невозможно🤢
  • Белое не отстирал😥
  • Ужасный запах🙈
  • Вкусный сыр👍
  • Песик не оценил😌
  • Всю кожу ободрала🥴
  • не сильно уксусные☹️
  • Развариваются знатно 😕 Но вкус хороший.
  • Цена соответствует качеству🙈
Рейтинг 4 — коротких эмодзи-отзывов

In [11]:
# удаляем отзывы «только эмодзи»: после очистки они станут пустыми (текстового сигнала нет)
emoji_only = has_emoji & ~df["review_text"].apply(
    lambda t: bool(re.search(r"[a-zа-яё]", strip_emoji(t).lower())))
print("Удаляем отзывов «только эмодзи»:", emoji_only.sum())
df = df[~emoji_only].reset_index(drop=True)
print("Осталось строк:", len(df))

Удаляем отзывов «только эмодзи»: 77
Осталось строк: 627544


**Вывод по эмодзи.** Эмодзи встречаются лишь в ~3.6 % отзывов, из них 99.7 % идут вместе с текстом, где оценка уже выражена словами («очень вкусно 😋», «отличный шоколад 👍»). Сдвиг к позитиву есть, но умеренный, а словесный сигнал дублирует эмодзи. Поэтому:
- отзывы **«только эмодзи»** (77 шт.) удаляем — текстового сигнала в них нет;
- у остальных эмодзи **удаляем при очистке** (см. `regex_clean`): полярность сохраняется в словах, усложнять пайплайн (demojize/отдельный признак) ради 3.6 % нецелесообразно.

## 4. Признак структурированности отзыва

Многие отзывы написаны по шаблону «Достоинства: … Недостатки: …». Сами слова-ярлыки неинформативны (стоят в любом отзыве) и уйдут в стоп-слова, но **факт** шаблона полезен: такие отзывы развёрнутые и чаще амбивалентные. Сохраняем его отдельным бинарным признаком `is_structured` — считаем **по сырому тексту**, до удаления ярлыков.

In [12]:
df["is_structured"] = (
    df["review_text"].str.contains(r"достоинств", case=False, na=False) &
    df["review_text"].str.contains(r"недостатк", case=False, na=False)
).astype(int)

print("Доля структурированных:", f"{df['is_structured'].mean():.2%}")
print("Средний рейтинг: шаблон =", round(df.loc[df.is_structured==1,"rating"].mean(),2),
      "| свободные =", round(df.loc[df.is_structured==0,"rating"].mean(),2))
print("Доля 3–4: шаблон =", f'{df.loc[df.is_structured==1,"rating"].isin([3,4]).mean():.2%}',
      "| свободные =", f'{df.loc[df.is_structured==0,"rating"].isin([3,4]).mean():.2%}')

Доля структурированных: 10.92%
Средний рейтинг: шаблон = 4.62 | свободные = 4.63
Доля 3–4: шаблон = 38.01% | свободные = 15.62%


Средний рейтинг у шаблонных и свободных отзывов почти совпадает, **но доля оценок 3–4 вдвое выше** (≈38 % против ≈16 %): шаблонные отзывы концентрируются в середине шкалы — это и есть самые трудные для классификатора классы. Признак слабый, но полезен как маркер амбивалентности.

## 5. Предобработка текста

Конвейер к `review_text`:
1. **Regex-очистка**: нижний регистр, удаление артефактов выгрузки (`_x000D_`), ссылок, эмодзи, цифр и пунктуации; оставляем только буквы (кириллица + латиница — латиница это бренды вроде `fairy`, `activia`).
2. **Токенизация** (`word_tokenize`).
3. **Удаление стоп-слов** — не базовый список, а адаптированный под тональность (см. ниже).
4. **Лемматизация** (`pymorphy3`).

**Логика стоп-слов:**
- **оставляем** (несут тональность): отрицания (`не, нет, ни, без…` — переворачивают смысл), прямые оценки (`хорошо, лучше`), степень (`больше, много, совсем, чуть…`);
- **добавляем** частотный шум без полярности (`весь, именно, такой, раз…`) — проверено по контекстам;
- **добавляем** шаблонные ярлыки формы (`достоинство, недостаток, плюс, минус, итог, вывод…`) — boilerplate, структуру уже сохранили в `is_structured`.

Числа удаляем полностью (не переводим в слова): иначе «пять»/«сто» из оценок в тексте («ставлю 5») становятся признаком, дублирующим целевую переменную.

In [13]:
from string import punctuation

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from pymorphy3 import MorphAnalyzer
import emoji
from tqdm.auto import tqdm
tqdm.pandas()

morph = MorphAnalyzer()
punctuations = set(punctuation)

# Стоп-слова
# оставляем (несут тональность): отрицания / оценка / степень / контраст
keep = {
    "не", "нет", "ни", "нельзя", "без", "ничего", "но",
    "хорошо", "лучше",
    "больше", "более", "много", "совсем", "почти", "чуть",
}

# nltk-список — поверхностные формы, фильтруем ДО лемматизации (как в лекции)
ru_stop_surface = set(stopwords.words("russian")) - keep

# наши добавки заданы ЛЕММАМИ, фильтруем ПОСЛЕ лемматизации
extra_stop = {"весь", "именно", "покупка", "один", "такой", "раз"}          # частотный шум без полярности
template_stop = {                                                            # шаблонные ярлыки формы отзыва
    "достоинство", "недостаток", "плюс", "минус",
    "преимущество", "итог", "вывод", "резюме", "вердикт", "комментарий",
}
custom_lemma_stop = extra_stop | template_stop

# Функции обработки
def lemmatize_word(w):
    return morph.parse(w)[0].normal_form

def regex_clean(text):
    t = str(text)
    t = re.sub(r"_x[0-9a-fA-F]{4}_", " ", t)   # артефакты Excel/OOXML (_x000D_ и т.п.)
    t = t.lower().replace("ё", "е")
    t = re.sub(r"https?://\S+", " ", t)        # ссылки
    t = emoji.replace_emoji(t, replace=" ")     # эмодзи
    t = re.sub(r"[^a-zа-я\s]", " ", t)         # только буквы; цифры и пунктуация убираются
    t = re.sub(r"\s+", " ", t).strip()         # двойные пробелы
    return t

### Применение конвейера

Функция `preprocess` объединяет шаги и возвращает список лемм. Поле `preprocessed` (строка лемм) пойдёт в векторизацию.

In [14]:
def preprocess(text):
    tokens = word_tokenize(regex_clean(text))
    #поверхностный фильтр: пунктуация, nltk-стоп-слова, короткие токены
    tokens = [t for t in tokens if t not in punctuations and t not in ru_stop_surface and len(t) > 1]
    #лемматизация
    lemmas = [lemmatize_word(t) for t in tokens]
    #доменный шум и шаблонные ярлыки (уже по леммам)
    return [l for l in lemmas if l not in custom_lemma_stop]

df["tokens_lemmas"] = df["review_text"].progress_apply(preprocess)
df["preprocessed"]  = df["tokens_lemmas"].apply(" ".join)

df[["review_text", "tokens_lemmas", "preprocessed"]].head()

  0%|          | 0/627544 [00:00<?, ?it/s]

,review_text,tokens_lemmas,preprocessed
0,"Самое вкусное протеиновое печенье с белковым суфле из всех, что есть на рынке на данный момент. ...","[самый, вкусный, протеиновый, печение, белков, суфле, рынок, данный, момент, без, неприятный, пр...",самый вкусный протеиновый печение белков суфле рынок данный момент без неприятный протеиновый пр...
1,"Веселая свечка, но нам пришла розовая, как на фото) буду искать дальше нейтрального цвета единичку.","[весёлый, свечка, но, мы, прийти, розовый, фото, быть, искать, далёкий, нейтральный, цвет, едини...",весёлый свечка но мы прийти розовый фото быть искать далёкий нейтральный цвет единичка
2,Хорошая яркая свечка,"[хороший, яркий, свечка]",хороший яркий свечка
3,"Ребенок в восторге от свечек! Она красная, кстати, не розовая. Что круто!","[ребёнок, восторг, свечка, красный, кстати, не, розовый, круто]",ребёнок восторг свечка красный кстати не розовый круто
4,"Хорошая рыба, вкусная, свежая. Жаль, что возят без охлаждающего льда.","[хороший, рыба, вкусный, свежий, жаль, возить, без, охлаждать, лёд]",хороший рыба вкусный свежий жаль возить без охлаждать лёд


In [15]:
# Топ-30 самых частых лемм после предобработки
counter = Counter()
for toks in df["tokens_lemmas"]:
    counter.update(toks)

print("Топ-30 слов (леммы):")
for w, c in counter.most_common(30):
    print(f"{w:15s} {c}")

Топ-30 слов (леммы):
не              221189
вкусный         170475
очень           167065
хороший         132280
вкус            82435
но              71769
отличный        62838
цена            43240
брать           42939
нет             39418
вкусно          38076
это             36656
хорошо          33794
понравиться     33657
приятный        32085
свежий          31691
запах           31112
качество        30136
нравиться       29800
без             28850
покупать        28367
самый           27257
рекомендовать   26389
ребёнок         22706
состав          22659
упаковка        22651
сладкий         22278
вода            20494
мягкий          19522
большой         19372


**Сколько текстов «схлопнулись» после предобработки.** Короткие документы (≤3 лемм) малоинформативны для BoW/TF-IDF, а пустые (0 лемм) дают нулевой вектор — это мотивирует отфильтровать их перед векторизацией (раздел 7).

In [16]:
n_lemmas = df["tokens_lemmas"].apply(len)

for thr in [0, 1, 2, 3]:
    print(f"≤{thr} лемм: {(n_lemmas <= thr).sum():>7} ({(n_lemmas <= thr).mean():.2%})")

print("\nмедиана лемм:", int(n_lemmas.median()), "| среднее:", round(n_lemmas.mean(), 1))

≤0 лемм:     599 (0.10%)
≤1 лемм:   34727 (5.53%)
≤2 лемм:  119840 (19.10%)
≤3 лемм:  190363 (30.33%)

медиана лемм: 6 | среднее: 8.8


### Удаление пустых после предобработки

599 отзывов после очистки и удаления стоп-слов стали пустыми (0 лемм) — это нулевые векторы, сигнала не несут. Удаляем их из `df`, чтобы `df`, матрицы `X` и целевая `y` были одной длины (удобно для моделирования). Короткие непустые отзывы (1–3 леммы) оставляем — они несут тональность.

In [17]:
empty = df["preprocessed"].str.strip() == ""
print("Удаляем пустых после предобработки:", int(empty.sum()))
df = df[~empty].reset_index(drop=True)
print("Осталось строк:", len(df))

Удаляем пустых после предобработки: 599
Осталось строк: 626945


In [18]:
key_no_author = ["product_id", "product_name", "product_category",
                 "product_price", "review_text"]

dup_mask = df.duplicated(subset=key_no_author, keep=False)
print("Строк в дублях (без автора):", dup_mask.sum())
print("Уникальных комбинаций:", df[dup_mask].drop_duplicates(subset=key_no_author).shape[0])
print("Лишних копий (к удалению):", df.duplicated(subset=key_no_author).sum())

Строк в дублях (без автора): 25342
Уникальных комбинаций: 8888
Лишних копий (к удалению): 16454


In [19]:
d = df[dup_mask].copy()
d["n_words"] = d["review_text"].astype(str).str.split().str.len()
print("Длина текста в этих дублях:")
print(d["n_words"].describe())
print("доля <3 слов:", f'{(d["n_words"] < 3).mean():.1%}')

# примеры: сравнить длинные (вероятно реальные дубли) и короткие (совпадения)
print("\nДлинные (>=6 слов) — вероятно повторы:")
for t in d.loc[d["n_words"] >= 6, "review_text"].head(5):
    print("  •", " ".join(str(t).split())[:100])
print("\nКороткие (<3 слов) — вероятно совпадения:")
for t in d.loc[d["n_words"] < 3, "review_text"].head(5):
    print("  •", " ".join(str(t).split())[:60])

Длина текста в этих дублях:
count    25342.000000
mean         2.051417
std          2.605828
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max        108.000000
Name: n_words, dtype: float64
доля <3 слов: 85.6%

Длинные (>=6 слов) — вероятно повторы:
  • Полный восторг! Нареканий к крему нет. Работает.
  • Полный восторг! Нареканий к крему нет. Работает.
  • Малютка безопасная и натуральная, специально адаптирована для детей, поэтому даю ее ребенку каждый д
  • Малютка безопасная и натуральная, специально адаптирована для детей, поэтому даю ее ребенку каждый д
  • Вкусная ряженка, ребёнок пьёт с удовольствием

Короткие (<3 слов) — вероятно совпадения:
  • Хорошая
  • Хорошая
  • Хороший сыр
  • Очень вкусный
  • Вкусный сыр


In [20]:
nw = df["review_text"].astype(str).str.split().str.len()
for thr in [3, 4, 5, 6]:
    m = df.duplicated(subset=key_no_author) & (nw >= thr)
    print(f"порог ≥{thr} слов: к удалению {m.sum()}")

порог ≥3 слов: к удалению 2060
порог ≥4 слов: к удалению 446
порог ≥5 слов: к удалению 266
порог ≥6 слов: к удалению 183


In [21]:
key_no_author = ["product_id", "product_name", "product_category",
                 "product_price", "review_text"]
nw = df["review_text"].astype(str).str.split().str.len()
long_dup = df.duplicated(subset=key_no_author) & (nw >= 4)

print("Всего длинных дублей:", long_dup.sum())
print("\nРаспределение по рейтингу:")
cmp = pd.DataFrame({
    "дубли (шт)":   df.loc[long_dup, "rating"].value_counts().sort_index(),
    "дубли (доля)": df.loc[long_dup, "rating"].value_counts(normalize=True).round(3),
    "весь корпус":  df["rating"].value_counts(normalize=True).round(3),
}).fillna(0)
print(cmp)

Всего длинных дублей: 446

Распределение по рейтингу:
        дубли (шт)  дубли (доля)  весь корпус
rating                                       
0              0.0         0.000        0.000
1              2.0         0.004        0.027
2              3.0         0.007        0.018
3              4.0         0.009        0.031
4             30.0         0.067        0.150
5            407.0         0.913        0.775


In [22]:
nw = df["review_text"].astype(str).str.split().str.len()
short_dup = df.duplicated(subset=key_no_author, keep=False) & (nw < 4)

print("Коротких дублей (строк):", short_dup.sum())
print("\nРаспределение по рейтингу:")
cmp = pd.DataFrame({
    "короткие дубли (шт)":   df.loc[short_dup, "rating"].value_counts().sort_index(),
    "короткие дубли (доля)": df.loc[short_dup, "rating"].value_counts(normalize=True).round(3),
    "весь корпус":           df["rating"].value_counts(normalize=True).round(3),
}).fillna(0)
print(cmp)

Коротких дублей (строк): 24470

Распределение по рейтингу:
        короткие дубли (шт)  короткие дубли (доля)  весь корпус
rating                                                         
0                       0.0                  0.000        0.000
1                      92.0                  0.004        0.027
2                     101.0                  0.004        0.018
3                     184.0                  0.008        0.031
4                    1766.0                  0.072        0.150
5                   22327.0                  0.912        0.775


### Удаление длинных текстовых дублей

И длинные, и короткие дубли распределены по рейтингу почти как весь корпус (~91 % — 5★), поэтому их удаление не меняет баланс классов. Короткие (<4 слов) — случайные совпадения generic-фраз («вкусно», «хороший сыр») от разных людей, это самостоятельные отзывы → **оставляем**. Длинные (≥4 слов) — дословные повторы целых фраз, почти наверняка копии → **удаляем**.

In [23]:
nw = df["review_text"].astype(str).str.split().str.len()
long_dup = df.duplicated(subset=key_no_author) & (nw >= 4)

print("Удаляем длинных текстовых дублей (без автора, ≥4 слов):", int(long_dup.sum()))
df = df[~long_dup].reset_index(drop=True)
print("Осталось строк:", len(df))

Удаляем длинных текстовых дублей (без автора, ≥4 слов): 446
Осталось строк: 626499


## 6. Обоснование выбора стоп-слов

Ниже — проверки на данных, которыми обоснованы решения выше. Контексты смотрим по **сырому** тексту (часть слов уже удалена из лемм).

### Спорные слова — просмотр контекстов

Для каждого слова смотрим реальные употребления: несёт оценку/усиление — оставляем; пустое указательное/счётное — в стоп-слова.

In [24]:
disputed = ["такой", "просто", "раз", "весь", "мало", "достаточно",
            "вполне", "вообще", "именно", "товар", "магазин", "производитель"]

for word in disputed:
    mask = df["review_text"].str.contains(rf"(?<![а-яё]){word[:-1]}", case=False, regex=True, na=False)
    print("=" * 70)
    print(f"«{word}» — встретилось примерно в {mask.sum()} отзывах")
    for t in df.loc[mask, "review_text"].head(5):
        print("  •", " ".join(str(t).split())[:120])

«такой» — встретилось примерно в 11932 отзывах
  • вот ровно так как я люблю сибаса, я не люблю чистить рыбу! А тут наконец такое счастье - филешки! Спасибо Впрок - вы чуд
  • Даёт цвет постепенно, не пятнит. Не такой явный как кларанс или st.tropez, но за такую цену вполне рабочий. С сухой коже
  • Вискас, как Сникерс, он и в Африке вискас. Им кормов такого типа- лучший.
  • Достоинства: Пить можно ) дешевый Недостатки: Не такой насыщенный вкус ! Больше похоже на компот
  • Приятное сочетание вкусов с небольшой кислинкой Мне такое не нравится, но девушка просто в восторге Так что, дело вкуса 
«просто» — встретилось примерно в 19546 отзывах
  • Рыбка понравилась: соль,лимон, в муку и на сковороду, быстро и просто, 2-3 маленькие косточки на 9 кусков, (столько было
  • Сделала на всё тело после скрабирования. Перед нанесением средства нанесла масло на кожу, средство втирала специальной р
  • Отличный автозагар. Если все делать по инструкции, то все просто замечательно.
  • Приятное сочет

По контекстам: `просто, мало, достаточно, вполне, вообще` несут оценку/степень → **оставляем**; `товар, магазин, производитель` — аспекты (объект оценки, нужны чтобы понять «к чему претензия») → **оставляем**; `такой, весь, именно, раз, один` — чаще служебные → **в стоп-слова**.

### Маркер контраста «однако»

Гипотеза: `однако` (и `но`, `хотя`, `зато`) маркирует амбивалентные отзывы середины шкалы. Проверяем распределение оценок.

In [26]:
mark = df["tokens_lemmas"].apply(lambda t: "однако" in t)
print("Доля отзывов с «однако»:", f"{mark.mean():.2%}")

cmp = pd.DataFrame({
    "все":        df["rating"].value_counts(normalize=True),
    "с «однако»": df.loc[mark, "rating"].value_counts(normalize=True),
}).fillna(0).sort_index().round(3)
print(cmp)
print("\nДоля 3–4: все =", f'{df["rating"].isin([3,4]).mean():.2%}',
      "| с «однако» =", f'{df.loc[mark, "rating"].isin([3,4]).mean():.2%}')

Доля отзывов с «однако»: 0.11%
          все  с «однако»
rating                   
0       0.000       0.000
1       0.027       0.031
2       0.018       0.013
3       0.031       0.077
4       0.150       0.434
5       0.775       0.444

Доля 3–4: все = 18.06% | с «однако» = 51.14%


In [27]:
# доля «однако» внутри каждой оценки
share = df.groupby("rating").apply(
    lambda g: g["tokens_lemmas"].apply(lambda t: "однако" in t).mean(), include_groups=False
).round(4)
print("Доля отзывов с «однако» внутри каждой оценки:")
print((share * 100).round(2).astype(str) + " %")

Доля отзывов с «однако» внутри каждой оценки:
rating
0     0.0 %
1    0.13 %
2    0.08 %
3    0.28 %
4    0.32 %
5    0.06 %
dtype: str


Доля 3–4 среди отзывов с «однако» подскакивает примерно с 18 % до 51 %, «проникновение» слова максимально у оценок 3–4. Гипотеза подтверждена → контрастные союзы (`однако, но, хотя, зато`) **оставляем** как слабый признак амбивалентности.

### Шаблонные ярлыки формы отзыва

Проверяем, какие слова реально используются как заголовки секций (перед `:` / тире). Частые заголовки — boilerplate, их в стоп-слова.

In [28]:
candidates = ["достоинства", "недостатки", "плюсы", "минусы", "плюс", "минус",
              "достоинство", "недостаток", "преимущества", "итог", "вывод",
              "резюме", "вердикт", "комментарий", "претензия", "дефект", "изъян"]

text = df["review_text"].dropna().astype(str)
rows = []
for w in candidates:
    as_header = text.str.contains(rf"(?:^|[\n.!?)])\s*{w}\s*[:\-–]", case=False, regex=True).sum()
    total = text.str.contains(re.escape(w), case=False, regex=True).sum()
    rows.append((w, int(as_header), int(total), round(as_header/total, 3) if total else 0))

res = (pd.DataFrame(rows, columns=["слово", "как_заголовок", "всего", "доля_заголовок"])
         .sort_values("как_заголовок", ascending=False))
print(res.to_string(index=False))

       слово  как_заголовок  всего  доля_заголовок
 достоинства         113843 114168           0.997
  недостатки          70094  70447           0.995
       плюсы           1693   2022           0.837
      минусы           1674   1866           0.897
 комментарий            278    455           0.611
       минус            138   5032           0.027
  недостаток             94    750           0.125
       вывод             86    921           0.093
        итог             56   1020           0.055
        плюс             36   5154           0.007
 достоинство             24    205           0.117
     вердикт             17     40           0.425
преимущества             11     53           0.208
      резюме              7     16           0.438
   претензия              0     65           0.000
      дефект              0    250           0.000
       изъян              0    286           0.000


In [29]:
# примеры употребления как заголовка
for w in ["достоинства", "недостатки", "плюсы", "минусы", "итог", "вывод"]:
    pat = rf"(?:^|[\n.!?)])\s*{w}\s*[:\-–]"
    hits = text[text.str.contains(pat, case=False, regex=True)]
    if len(hits) == 0:
        continue
    print("=" * 70)
    print(f"«{w}» — как заголовок в {len(hits)} отзывах")
    for t in hits.head(3):
        print("  •", " ".join(t.split())[:120])

«достоинства» — как заголовок в 113843 отзывах
  • Достоинства: Норм, кот ест корм. Недостатки: 75гр #корм
  • Достоинства: Котята с удовольствием кушают маленькие кусочки едят с аппетитом Рекомендую
  • Достоинства: Малышке😻очень нравится whiskas для котят, в детском варианте маленькие кусочки, для малышки удобно👍Ест с уд
«недостатки» — как заголовок в 70094 отзывах
  • Достоинства: Норм, кот ест корм. Недостатки: 75гр #корм
  • Достоинства: Малышке😻очень нравится whiskas для котят, в детском варианте маленькие кусочки, для малышки удобно👍Ест с уд
  • Достоинства: Цена, запах, долго держится Недостатки: Минусов пока не нашла
«плюсы» — как заголовок в 1693 отзывах
  • Плюсы: + тонкие (не мешают, не шуршат и вообще очень удобно 😁) + хорошая клейкая основа (ни разу не отрывались без моего
  • Я просто обожаю подобные рецепты, они намного быстрее приготовить обед или ужин и при этом ещё будет очень вкусно Плюсы:
  • Плюсы: слабогазированная. На вкус слегка химическая, пить можно, но больш

`достоинства`/`недостатки` используются как заголовки в десятках тысяч отзывов, `плюсы`/`минусы`/`итог`/`вывод`/`резюме`/`вердикт`/`комментарий` — тоже шаблонные ярлыки. Сам токен полярности не несёт (оценка в соседнем слове) → **в стоп-слова** (по леммам, что покрывает и мн. число). А слова `претензия, дефект, изъян` как заголовки не используются и осмысленны в тексте → **оставляем**. Структурную роль мы уже сохранили в `is_structured` (раздел 4).

## 7. Разбиение на train / validation / test

Разбиваем **по `product_id`** со стратификацией по оценке (`StratifiedGroupKFold`), как в ДЗ 1: все отзывы одного товара целиком попадают в одну выборку. Иначе отзывы одного товара окажутся и в train, и в test — это утечка, и оценка качества будет завышена. Соотношение 70 / 15 / 15.

**Почему именно до векторизации.** `CountVectorizer`/`TfidfVectorizer` при `fit` строят словарь и считают IDF. Если обучить их на всём корпусе, статистика из test просочится в признаки train. Поэтому `fit` делаем **только на train**, а к val/test применяем `transform` тем же словарём и IDF.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

y = df["rating"].to_numpy()
groups = df["product_id"].to_numpy()

# test ≈ 1/7 ≈ 14%
sgkf1 = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(sgkf1.split(df, y, groups))

# из оставшихся отделяем val ≈ 1/6 ≈ 14% от всего
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=RANDOM_STATE)
tr_rel, val_rel = next(sgkf2.split(df.iloc[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_rel], trainval_idx[val_rel]

train_text = df.iloc[train_idx]["preprocessed"]
val_text   = df.iloc[val_idx]["preprocessed"]
test_text  = df.iloc[test_idx]["preprocessed"]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print("train / val / test:", len(train_idx), len(val_idx), len(test_idx))

# проверка отсутствия утечки по товарам (должны быть нули)
print("пересечения товаров train/val, train/test, val/test:",
      len(set(groups[train_idx]) & set(groups[val_idx])),
      len(set(groups[train_idx]) & set(groups[test_idx])),
      len(set(groups[val_idx])   & set(groups[test_idx])))

# доли оценок по выборкам (стратификация сохраняет пропорции)
print("\nДоли оценок:")
print(pd.DataFrame({
    "train": pd.Series(y_train).value_counts(normalize=True),
    "val":   pd.Series(y_val).value_counts(normalize=True),
    "test":  pd.Series(y_test).value_counts(normalize=True),
}).sort_index().round(3))

## 8. Векторизация: BoW и TF-IDF

Предобработанные тексты переводим в числовые векторы двумя способами. Векторайзеры **обучаем только на train** (`fit_transform`), а к val/test применяем `transform` — так словарь и IDF не «подсматривают» тестовые данные.

In [30]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# словарь и веса строим по train; corpus = train для примеров ниже
corpus = train_text
print("Документов для обучения векторайзеров (train):", len(corpus))

Непустых документов для векторизации: 626499 (пустых после предобработки: 0)


### Bag of Words

`CountVectorizer` строит словарь и считает частоту каждого слова в документе. `min_df=5` отсекает крайне редкие слова (опечатки, единичные бренды), чтобы словарь был разумного размера.

In [31]:
bow_vectorizer = CountVectorizer(min_df=5)
X_bow_train = bow_vectorizer.fit_transform(train_text)   # fit ТОЛЬКО на train
X_bow_val   = bow_vectorizer.transform(val_text)
X_bow_test  = bow_vectorizer.transform(test_text)

print("BoW train/val/test:", X_bow_train.shape, X_bow_val.shape, X_bow_test.shape)
print("Размер словаря:", len(bow_vectorizer.get_feature_names_out()))

Матрица BoW: (626499, 77520) | тип: csr_matrix
Размер словаря: 77520


### TF-IDF

`TfidfVectorizer` взвешивает слова: частые в документе, но редкие в корпусе получают больший вес, а повсеместные (как «вкусный») — меньший. Словарь тот же, что у BoW (`min_df=5`).

In [32]:
tfidf_vectorizer = TfidfVectorizer(min_df=5)
X_tfidf_train = tfidf_vectorizer.fit_transform(train_text)   # fit ТОЛЬКО на train
X_tfidf_val   = tfidf_vectorizer.transform(val_text)
X_tfidf_test  = tfidf_vectorizer.transform(test_text)

print("TF-IDF train/val/test:", X_tfidf_train.shape, X_tfidf_val.shape, X_tfidf_test.shape)

Матрица TF-IDF: (626499, 20033) | тип: csr_matrix


### Сравнение представлений

In [33]:
def describe(name, X):
    rows, cols = X.shape
    nnz = X.nnz
    sparsity = 1 - nnz / (rows * cols)
    print(f"{name}:")
    print(f"  размерность:        {rows} × {cols}")
    print(f"  ненулевых элементов: {nnz:,}")
    print(f"  разреженность:       {sparsity:.4%}")
    print(f"  в среднем слов/док:  {nnz / rows:.1f}")

describe("BoW (train)", X_bow_train)
print()
describe("TF-IDF (train)", X_tfidf_train)

BoW:
  размерность:        626499 × 77520
  ненулевых элементов: 5,193,720
  разреженность:       99.9893%
  в среднем слов/док:  8.3

TF-IDF:
  размерность:        626499 × 20033
  ненулевых элементов: 5,109,915
  разреженность:       99.9593%
  в среднем слов/док:  8.2


Обе матрицы имеют одинаковую размерность (число документов × размер словаря) и крайне **разрежены** (>99.9 % нулей): документ содержит лишь малую часть словаря. Поэтому они хранятся в sparse-формате.

In [34]:
# примеры документов и их векторов (ненулевые признаки), на train
bow_terms = bow_vectorizer.get_feature_names_out()
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

for i in range(3):
    print("=" * 70)
    print("Документ:", corpus.iloc[i][:100])
    row_b = X_bow_train[i]
    print("  BoW   :", {bow_terms[j]: int(row_b[0, j]) for j in row_b.indices})
    row_t = X_tfidf_train[i]
    print("  TF-IDF:", {tfidf_terms[j]: round(float(row_t[0, j]), 3) for j in row_t.indices})

Документ: самый вкусный протеиновый печение белков суфле рынок данный момент без неприятный протеиновый привку
  BoW   : {'самый': 1, 'вкусный': 1, 'протеиновый': 2, 'печение': 1, 'белков': 1, 'суфле': 1, 'рынок': 1, 'данный': 1, 'момент': 1, 'без': 1, 'неприятный': 1, 'привкус': 1, 'присущий': 1, 'подобный': 1, 'род': 1, 'изделие': 1}
  TF-IDF: {'самый': 0.12, 'вкусный': 0.068, 'протеиновый': 0.571, 'печение': 0.17, 'белков': 0.338, 'суфле': 0.246, 'рынок': 0.208, 'данный': 0.18, 'момент': 0.214, 'без': 0.12, 'неприятный': 0.192, 'привкус': 0.172, 'присущий': 0.291, 'подобный': 0.216, 'род': 0.258, 'изделие': 0.227}
Документ: весёлый свечка но мы прийти розовый фото быть искать далёкий нейтральный цвет единичка
  BoW   : {'весёлый': 1, 'свечка': 1, 'но': 1, 'мы': 1, 'прийти': 1, 'розовый': 1, 'фото': 1, 'быть': 1, 'искать': 1, 'далёкий': 1, 'нейтральный': 1, 'цвет': 1, 'единичка': 1}
  TF-IDF: {'весёлый': 0.326, 'свечка': 0.414, 'но': 0.12, 'мы': 0.21, 'прийти': 0.254, 'розовый': 0.28

### Чем отличаются BoW и TF-IDF

| | Bag of Words | TF-IDF |
|---|---|---|
| Значение ячейки | частота слова в документе (целое) | вес = TF × IDF (дробный) |
| Учёт редкости слова | нет | да — редкие в корпусе слова весят больше |
| Частые слова («вкусный») | большой вес | приглушены низким IDF |
| Нормировка длины | нет | да (L2 по умолчанию) |

**Вывод.** BoW отражает, *сколько раз* слово встретилось, одинаково оценивая и повсеместные, и редкие слова. TF-IDF дополнительно учитывает *информативность* слова в корпусе: общеупотребительные слова получают меньший вес, характерные — больший, а векторы нормируются по длине документа. Для классификации тональности TF-IDF обычно предпочтительнее, так как снижает влияние «фоновой» лексики и подчёркивает отличительные слова. Размерность пространства у обоих методов одинакова — различается лишь способ вычисления координат.